# Notebook 4: Out-of-Sample Backtest & Quant Performance Report

Trong notebook này, chúng ta thực hiện kiểm định (Backtest) chiến lược trên dữ liệu hoàn toàn chưa biết (Out-of-Sample từ `2023-07-01` đến `2024-06-30`):
1. Mô phỏng tái cơ cấu danh mục hàng tuần/hàng kỳ với chi phí giao dịch và thuế (`0.20%/trade`).
2. Tính toán bảng KPI tài chính định lượng chuyên nghiệp: **Sharpe Ratio**, **Maximum Drawdown**, **CAGR**, **Information Ratio**.
3. Trực quan hóa biểu đồ chuẩn xuất bản: Equity Curve, Underwater Drawdowns, Rolling Beta, và Allocation History.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.data_pipeline import DataFetcher, DataCleaner, FeatureEngineer, DataPreprocessor
from src.ai_models import create_dataloaders, AlphaMLP, AlphaTrainer, AlphaPredictor
from src.risk_models import RiskModel, BetaCalculator
from src.optimization import PortfolioOptimizer
from src.backtest import BacktestEngine, PerformanceEvaluator, BacktestVisualizer

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Chuẩn bị Pipeline và Mô hình AI đã huấn luyện


In [ ]:
fetcher = DataFetcher(use_mock_fallback=True)
raw_data = fetcher.fetch_all()
cleaned_data = DataCleaner().clean_and_align(raw_data)
feature_data = FeatureEngineer().compute_all_features(cleaned_data)
preprocessor = DataPreprocessor()
normalized_data = preprocessor.normalize_features(feature_data)

master_df = preprocessor.prepare_tabular_dataset(normalized_data, Config.START_DATE, Config.END_DATE)
train_loader, val_loader, test_loader, test_df = create_dataloaders(master_df, batch_size=64)

model = AlphaMLP(input_dim=len(FeatureEngineer.get_feature_names()))
trainer = AlphaTrainer(model, lr=1e-3)
history = trainer.fit(train_loader, val_loader, epochs=15)

predictor = AlphaPredictor(model)
test_df_with_preds = predictor.predict_all(test_df)
for sym in normalized_data.keys():
    if sym == Config.BENCHMARK_TICKER:
        continue
    sym_preds = test_df_with_preds[test_df_with_preds['symbol'] == sym].set_index('date')['predicted_mu']
    normalized_data[sym] = normalized_data[sym].merge(sym_preds, on='date', how='left')
    normalized_data[sym]['predicted_mu'] = normalized_data[sym]['predicted_mu'].ffill().fillna(0.0)


## 2. Thực thi Backtest Engine Out-of-Sample (Tái cơ cấu Hàng tuần)


In [ ]:
engine = BacktestEngine(
    predictor=predictor,
    risk_model=RiskModel(),
    beta_calc=BetaCalculator(),
    optimizer=PortfolioOptimizer(),
    rebalance_freq='weekly',
    fee_rate=0.0020
)

backtest_res = engine.run(cleaned_data, normalized_data, start_date=Config.TEST_START_DATE, end_date=Config.END_DATE)
results_df = backtest_res['results_df']
weights_df = backtest_res['weights_df']
print('Hoàn tất Backtest Out-of-Sample!')


## 3. Báo cáo Tóm tắt Hiệu suất (KPI Summary Table)


In [ ]:
evaluator = PerformanceEvaluator()
summary_df = evaluator.evaluate(results_df)
display(summary_df)


## 4. Biểu đồ Đường cong Tài sản & Quản trị Rủi ro


In [ ]:
visualizer = BacktestVisualizer(output_dir=Path('notebook_charts'))

# 1. Equity Curves
visualizer.plot_equity_curves(results_df)
plt.show()

# 2. Drawdowns
visualizer.plot_drawdowns(results_df)
plt.show()

# 3. Rolling Beta Verification
visualizer.plot_rolling_beta(results_df)
plt.show()

# 4. Long/Short Exposure
visualizer.plot_exposure_history(weights_df)
plt.show()
